# Amplitude REST Chart Extractor v11
### Saved chart URL -> API -> DataFrame
---

## 1. Setup

In [36]:
import httpx, os, re, json, io, csv
import pandas as pd
from datetime import datetime
from base64 import b64encode

## 2. REST Auth

In [37]:
# ══════════════════════════════════════════════════════════
# REST API Auth (requerido)
# ══════════════════════════════════════════════════════════
_secrets = {}

def _get_secret(name):
    if name in _secrets:
        return _secrets[name]

    val = None
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        pass

    if not val:
        try:
            from dotenv import load_dotenv
            load_dotenv()
            val = os.environ.get(name)
        except ImportError:
            pass

    if val:
        _secrets[name] = val
    return val

_api_key = _get_secret("AMPLITUDE_API_KEY")
_api_secret = _get_secret("AMPLITUDE_API_SECRET")

if not _api_key or not _api_secret:
    raise SystemExit("⚠️ Faltan AMPLITUDE_API_KEY / AMPLITUDE_API_SECRET en Secrets o .env.")

_rest_auth = b64encode(f"{_api_key}:{_api_secret}".encode()).decode()
REST_HEADERS = {"Authorization": f"Basic {_rest_auth}"}

print("REST API:  ✅ Credenciales cargadas")

REST API:  ✅ Credenciales cargadas


## 3. API Helpers

In [38]:
print("REST-only mode active.")

REST-only mode active.


## 4. Utilities

In [39]:
# ══ Utility Functions (REST only) ══

def _to_api_date(date_text):
    # Convierte YYYY-MM-DD a YYYYMMDD para la API
    datetime.strptime(date_text, "%Y-%m-%d")
    return date_text.replace("-", "")


def extract_chart_id(chart_url):
    # Acepta solo saved charts; chart/new requiere guardarlo primero
    match_edit = re.search(r"/chart/new/([a-zA-Z0-9]+)", chart_url)
    if match_edit:
        raise ValueError(
            "La URL corresponde a un chart edit (chart/new). Guarda el chart y usa su URL tipo /chart/{id}."
        )

    match_saved = re.search(r"/chart/([a-zA-Z0-9]+)(?:\?|$|/)", chart_url)
    if not match_saved:
        raise ValueError(f"No se pudo extraer chart_id desde la URL: {chart_url}")

    return match_saved.group(1)


def _decode_series_label(item):
    # Normaliza labels con formatos mixtos en la respuesta
    if isinstance(item, str):
        return item

    if isinstance(item, dict):
        if "setName" in item and item["setName"]:
            return str(item["setName"])
        return str(item)

    if isinstance(item, list):
        if len(item) == 2 and isinstance(item[1], str):
            return item[1]
        if item and isinstance(item[0], dict) and item[0].get("setName"):
            return str(item[0]["setName"])
        return " | ".join(str(x) for x in item)

    return str(item)


def _split_label_parts(label_text):
    # Separa combinaciones de group by que pueden venir con coma o punto y coma
    if "; " in label_text:
        return [part.strip() for part in label_text.split("; ") if part.strip()]
    if ", " in label_text:
        return [part.strip() for part in label_text.split(", ") if part.strip()]
    return [label_text.strip()] if label_text else ["(none)"]


def _clean_text_token(value):
    # Limpia tabs escapados/reales y saltos de linea en strings
    if not isinstance(value, str):
        return value

    cleaned = value.replace("\\t", " ").replace("\t", " ")
    cleaned = cleaned.replace("\\n", " ").replace("\n", " ")
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip()


def _clean_dataframe_strings(df):
    if df.empty:
        return df

    # Limpieza de nombres de columnas
    df.columns = [_clean_text_token(col) for col in df.columns]

    # Limpieza de columnas string
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in obj_cols:
        df[col] = df[col].map(_clean_text_token)

    return df


def _normalize_data_block(raw_data):
    # Normaliza data para soportar dict, JSON-string y CSV-string
    if isinstance(raw_data, dict):
        return raw_data

    if isinstance(raw_data, str):
        candidate = raw_data.strip()
        if not candidate:
            return {}

        try:
            parsed = json.loads(candidate)
            return parsed if isinstance(parsed, dict) else {"_raw_value": parsed}
        except json.JSONDecodeError:
            return {"_raw_csv": candidate}

    return {}


def _parse_csv_payload(csv_text):
    # Fallback para respuestas tipo CSV embebido en string
    raw_text = str(csv_text or "").strip()
    if not raw_text:
        return pd.DataFrame()

    # Convierte secuencias escapadas para parsear correctamente
    normalized = raw_text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\t", "\t")

    try:
        rows = list(csv.reader(io.StringIO(normalized), delimiter=",", quotechar='"'))
    except Exception:
        return pd.DataFrame()

    # Quita filas completamente vacias
    cleaned_rows = []
    for row in rows:
        if not row:
            continue
        if all((cell is None) or (str(cell).strip() == "") for cell in row):
            continue
        cleaned_rows.append(row)

    if not cleaned_rows:
        return pd.DataFrame()

    # Busca header real: primera fila con 2+ celdas no vacias
    header_idx = None
    for idx, row in enumerate(cleaned_rows):
        non_empty = [cell for cell in row if str(cell).strip() != ""]
        if len(non_empty) >= 2:
            header_idx = idx
            break

    if header_idx is None:
        # Caso degradado: una sola columna
        values = [str(row[0]) if row else "" for row in cleaned_rows]
        df = pd.DataFrame({"value": values})
        return _clean_dataframe_strings(df)

    header_row = cleaned_rows[header_idx]
    n_cols = len(header_row)

    # Normaliza longitud de filas para construir tabla estable
    data_rows = []
    for row in cleaned_rows[header_idx + 1 :]:
        row_extended = list(row) + [""] * max(0, n_cols - len(row))
        row_trimmed = row_extended[:n_cols]
        if all(str(cell).strip() == "" for cell in row_trimmed):
            continue
        data_rows.append(row_trimmed)

    if not data_rows:
        return pd.DataFrame()

    df = pd.DataFrame(data_rows, columns=header_row)

    # Remueve columnas totalmente vacias
    keep_cols = [col for col in df.columns if not df[col].astype(str).str.strip().eq("").all()]
    df = df[keep_cols] if keep_cols else df

    df = _clean_dataframe_strings(df)

    # Renombra columnas vacias o duplicadas de forma estable
    renamed_cols = []
    seen = {}
    for idx, col in enumerate(df.columns):
        base = col if col else f"group_{idx+1}"
        count = seen.get(base, 0)
        final_name = base if count == 0 else f"{base}_{count+1}"
        seen[base] = count + 1
        renamed_cols.append(final_name)
    df.columns = renamed_cols

    # Forward-fill en columnas categóricas para filas jerarquicas
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols) > 0:
        for col in obj_cols:
            df[col] = df[col].replace("", pd.NA)
        df[obj_cols] = df[obj_cols].ffill()

    return df.reset_index(drop=True)


def fetch_chart(chart_id, start_date, end_date, timeout=120):
    params = {
        "start": _to_api_date(start_date),
        "end": _to_api_date(end_date),
    }
    url = f"https://amplitude.com/api/3/chart/{chart_id}/csv"

    with httpx.Client(timeout=timeout) as client:
        response = client.get(url, params=params, headers=REST_HEADERS)

    if response.status_code != 200:
        raise RuntimeError(f"Amplitude API error {response.status_code}: {response.text[:500]}")

    try:
        return response.json()
    except Exception:
        return {"data": response.text}


def parse_response(payload):
    if not isinstance(payload, dict):
        return pd.DataFrame()

    data = _normalize_data_block(payload.get("data", {}))

    if "_raw_csv" in data:
        return _parse_csv_payload(data["_raw_csv"])

    series = data.get("series", []) if isinstance(data, dict) else []
    raw_labels = (data.get("seriesLabels") or data.get("seriesMeta") or []) if isinstance(data, dict) else []
    x_values = data.get("xValues", []) if isinstance(data, dict) else []

    if not series:
        return pd.DataFrame()

    decoded_labels = [_decode_series_label(raw_labels[i]) if i < len(raw_labels) else f"series_{i}" for i in range(len(series))]
    parts_by_series = [_split_label_parts(lbl) for lbl in decoded_labels]
    max_parts = max((len(parts) for parts in parts_by_series), default=1)

    group_cols = [f"group_{i+1}" for i in range(max_parts)]
    rows = []

    has_time = bool(x_values)
    for idx, serie in enumerate(series):
        parts = parts_by_series[idx]
        padded_parts = parts + ["(none)"] * (max_parts - len(parts))

        if has_time:
            for point_idx, value in enumerate(serie):
                if not isinstance(value, (int, float)):
                    continue
                raw_date = x_values[point_idx] if point_idx < len(x_values) else None
                date_value = str(raw_date)[:10] if raw_date is not None else None
                rows.append([date_value, *padded_parts, decoded_labels[idx], float(value)])
        else:
            total = sum(v for v in serie if isinstance(v, (int, float)))
            rows.append([*padded_parts, decoded_labels[idx], float(total)])

    columns = (["date"] if has_time else []) + group_cols + ["series_label", "value"]
    df = pd.DataFrame(rows, columns=columns)

    if has_time:
        df = df.sort_values(["date", "series_label"]).reset_index(drop=True)
    else:
        df = df.sort_values(["series_label"]).reset_index(drop=True)

    return _clean_dataframe_strings(df)


def extract_x_values(payload):
    # Lee xValues de forma robusta para logs de validacion
    if not isinstance(payload, dict):
        return []
    data = _normalize_data_block(payload.get("data", {}))
    if isinstance(data, dict):
        x_values = data.get("xValues", [])
        return x_values if isinstance(x_values, list) else []
    return []


print("REST utilities defined")

REST utilities defined


## 5. Extraccion

In [40]:
def extract(chart_url, start_date, end_date):
    chart_id = extract_chart_id(chart_url)
    print(f"\n{'='*60}")
    print(f"Chart ID:   {chart_id}")
    print(f"Rango:      {start_date} > {end_date}")
    print("Metodo:     Dashboard REST API (/api/3/chart/{id}/csv)")

    payload = fetch_chart(chart_id, start_date, end_date)
    df = parse_response(payload)

    if df.empty:
        print("Resultado:  sin datos")
        print(f"{'='*60}")
        return df

    x_values = extract_x_values(payload)
    if x_values:
        print(f"xValues API: {str(x_values[0])[:10]} > {str(x_values[-1])[:10]} ({len(x_values)} puntos)")

    print(f"Resultado:  {len(df)} filas x {len(df.columns)} columnas")
    print(f"Columnas:   {list(df.columns)}")
    print(f"{'='*60}")
    return df


print("extract() REST-only defined")

extract() REST-only defined


## 6. Configuracion y ejecucion

In [41]:
# ══════════════════════════════════════════════════════════
# CONFIGURACION (saved chart URL)
# ══════════════════════════════════════════════════════════
# Debe ser URL tipo /chart/{id}. No usar /chart/new/{id}.
CHART_URL = "https://app.amplitude.com/analytics/migrante/chart/emoy772d"

START_DATE = "2026-04-17"
END_DATE = "2026-04-17"

# START_DATE = "2026-03-01"
# END_DATE = "2026-03-31"

# ══════════════════════════════════════════════════════════
df = extract(CHART_URL, START_DATE, END_DATE)
if not df.empty:
    display(df.head(20))


Chart ID:   emoy772d
Rango:      2026-04-17 > 2026-04-17
Metodo:     Dashboard REST API (/api/3/chart/{id}/csv)
Resultado:  101 filas x 6 columnas
Columnas:   ['model', 'year', 'color', 'price', 'brand', 'Product Verification Status Updated--All Users']


,model,year,color,price,brand,Product Verification Status Updated--All Users
0,Pulsar N 160,2026,(none),37999,Bajaj,6.0
1,Pulsar N 160,2026,(none),38599,Bajaj,2.0
2,Pulsar N 160,2026,(none),37099,Bajaj,1.0
3,Pulsar N 160,2026,(none),53999,Bajaj,1.0
4,Pulsar N 250 FI ABS,2026,(none),66499,Bajaj,6.0
5,Pulsar N 250 FI ABS,2026,(none),69999,Bajaj,1.0
6,Pulsar N 250 FI ABS,2026,Blanco,66499,Bajaj,1.0
7,Pulsar N 125 FI CBS,2026,(none),37499,Bajaj,4.0
8,Pulsar N 125 FI CBS,2026,(none),36499,Bajaj,2.0
9,Pulsar N 125 FI CBS,2026,Purpura,37499,Bajaj,1.0


In [ ]:
df.to_csv(f'../src/data/amplitude/output/{START_DATE.replace("-", "")}-{END_DATE.replace("-", "")}-Amplitude_facturas.csv', index = False)

: 

## 7. Exportar (opcional)

In [ ]:
if not df.empty:
    filename = f"amplitude_{START_DATE}_{END_DATE}.csv"
    df.to_csv(filename, index=False)
    print(f"Exportado: {filename}")
    from google.colab import files
    files.download(filename)

In [ ]:
# Bloque legacy removido: usar solo la celda de configuracion anterior.